In [1]:
from langchain_community.document_loaders import WikipediaLoader, Docx2txtLoader, PyPDFLoader, TextLoader, CSVLoader, JSONLoader
from langchain_community.document_loaders import DirectoryLoader
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from concurrent.futures import ThreadPoolExecutor
from langchain_community.utilities import SQLDatabase
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool
from langchain.chains import create_sql_query_chain
from langchain_core.output_parsers import StrOutputParser

In [2]:
import os

local_llm = ChatOpenAI(
            api_key="ai",
            model="openai/gpt-oss-20b",
            base_url="http://192.168.0.110:8000/v1",
        )
prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    """
                    - 역할: 너는 최고의 번역가야
                    
                    - 요구사항
                    1. 제시되는 문장을 의미가 최대한 생략되지 않도록 영어에서 한글로 번역해줘
                    2. 그리고 도메인은 미군이야 최대한 군사용어로 의미가 바뀌지 않도록 번역해줘
                    3. 머리말과 꼬리말은 빼줘
                    머리말 예시1:
                    chapter4
                    ------------------------------------

                    머리말 예시2:
                    Battle Drills
                    ------------------------------------

                    머리말 예시3:
                    Introduction
                    ------------------------------------

                    머리말 예시4:
                               North Korean Fundamentals
                    ------------------------------------

                    꼬리말 예시1:
                    24 July 2020 ATP 7-100.2 6-15

                    꼬리말 예시2:
                    4-40 ATP 7-100.2 24 July 2020

                    꼬리말 예시3:
                    3-8 ATP 7-100.2 24 July 2020

                    꼬리말 예시4:
                    24 July 2020 ATP 7-100.2 8-9
                    """
                    # self.current_chat_room.m_prompt
                ),
                ("human", "{input}"),
            ]
        )
chain = (
    prompt
    | local_llm
)



path  = "./file/"
lst = os.listdir(path)
print(lst)
chunks = []
final = []
for i in lst:
    file = os.path.join(path, i)
    loader = PyPDFLoader(file)
    docs = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=4000, chunk_overlap=0)
    raw_chunks = text_splitter.split_documents(docs)

    for d in raw_chunks:
        chunks.append(d.page_content)

def run(docs):
    res = chain.invoke({"input": docs})
    return res

with ThreadPoolExecutor(max_workers=100) as executor:
    for docs in executor.map(run, chunks):
        final.extend(docs)

print(final)

        

['North_Korean_Tactics_ch4.pdf']
[('content', '**PART TWO**  \n*북한 행동*  \n\n**제2부**  \nKPA(북한 인민군) 육군(KPAGF) 정찰 및 보안, 공격, 방어, 반안정화(카운터스테이빌리티) 작전의 전술 과제 및 훈련을 다룬다. 전자정보전(EIW)은 모든 금해군(KPA) 작전의 필수 요소이다. 각 장은 순차, 병렬 또는 동시 행위의 유형을 전달하기 위한 지정된 과제, 하과제, 전술 작전 모델을 제시한다.\n\n**제4장**  \n*전투 훈련*  \n\n이 장에서는 미션 수행 중 상황 및 상황 신호에 따라 기반으로 한 KPAGF 전술 과제의 표준화된 실행을 기술한다. 전투 훈련의 목적에 관한 논의 후, 다루어지는 행위는 접촉 시 대응, 적군 고정, 접촉 파괴, 상황 돌파 실행, 그리고 화력과 기동이다. 이러한 드릴 수행 중에는 지휘·통제·보조(Combined Arms) 행위가 보완적으로 발생한다. 예를 들어 방해 행위, 전전 기획 항공 방어(AAAD), 항공 습격 방어 행위 등이 있다. 장은 복합 지형 작전으로 마무리된다.\n\n**전투 훈련의 목적**  \n\n4‑1. 전투 훈련은 상황 신호에 의해 발동되는 집단 과제이며, 지휘관의 최소 주문만으로 수행할 수 있다. 이는 미션 중에 갑자기 나타날 수 있는 조건에 반사적으로 대응하도록 사전에 리허설한 통합 행위로 구성된다. 그리고 바람직한 결과를 낸다. 전투 훈련은 분리무기 수준 이하에서 수행되는 부대 전술 과제이며, 다양한 하과제를 수행하는 기능별 조직화된 부대로 수행된다. 이러한 부대의 구성은 부대의 종류 및 작전 환경(OE)에 따라 달라지지만, 수행되는 하과제는 모든 전술 부대에 동일하다. 대부분의 전투 훈련은 대형 전술 미션 과제의 주행을 용이하게 하는 기능적 지원에 초점을 맞춘다.\n\n> 참고: KPA 문헌에는 “전투 훈련”에 대한 특정 용어가 없으며, 육군은 정찰 및 보안, 공격, 방어, 반안정화에 공통적인 전술 훈련을 리허설한다.\n\n

In [3]:
content = []
for i in final:
    if i[0] == 'content':
        content.append(i[1])
full_content = "\n".join(content)
print(full_content)

**PART TWO**  
*북한 행동*  

**제2부**  
KPA(북한 인민군) 육군(KPAGF) 정찰 및 보안, 공격, 방어, 반안정화(카운터스테이빌리티) 작전의 전술 과제 및 훈련을 다룬다. 전자정보전(EIW)은 모든 금해군(KPA) 작전의 필수 요소이다. 각 장은 순차, 병렬 또는 동시 행위의 유형을 전달하기 위한 지정된 과제, 하과제, 전술 작전 모델을 제시한다.

**제4장**  
*전투 훈련*  

이 장에서는 미션 수행 중 상황 및 상황 신호에 따라 기반으로 한 KPAGF 전술 과제의 표준화된 실행을 기술한다. 전투 훈련의 목적에 관한 논의 후, 다루어지는 행위는 접촉 시 대응, 적군 고정, 접촉 파괴, 상황 돌파 실행, 그리고 화력과 기동이다. 이러한 드릴 수행 중에는 지휘·통제·보조(Combined Arms) 행위가 보완적으로 발생한다. 예를 들어 방해 행위, 전전 기획 항공 방어(AAAD), 항공 습격 방어 행위 등이 있다. 장은 복합 지형 작전으로 마무리된다.

**전투 훈련의 목적**  

4‑1. 전투 훈련은 상황 신호에 의해 발동되는 집단 과제이며, 지휘관의 최소 주문만으로 수행할 수 있다. 이는 미션 중에 갑자기 나타날 수 있는 조건에 반사적으로 대응하도록 사전에 리허설한 통합 행위로 구성된다. 그리고 바람직한 결과를 낸다. 전투 훈련은 분리무기 수준 이하에서 수행되는 부대 전술 과제이며, 다양한 하과제를 수행하는 기능별 조직화된 부대로 수행된다. 이러한 부대의 구성은 부대의 종류 및 작전 환경(OE)에 따라 달라지지만, 수행되는 하과제는 모든 전술 부대에 동일하다. 대부분의 전투 훈련은 대형 전술 미션 과제의 주행을 용이하게 하는 기능적 지원에 초점을 맞춘다.

> 참고: KPA 문헌에는 “전투 훈련”에 대한 특정 용어가 없으며, 육군은 정찰 및 보안, 공격, 방어, 반안정화에 공통적인 전술 훈련을 리허설한다.

4‑2. 전투 훈련은 공격, 방어, 반안정화 작전에서 수행된다. 그 목적은 전투 속도를 제어함으로써 이득을 얻는 것

In [4]:
f = open('north_korean_tactics_ch4.txt', 'w', encoding="utf-8")
f.write(full_content)
f.close()

In [52]:
db = SQLDatabase.from_uri("sqlite:///D:\AI_team\sqlLite\devDB.db")
print(db.get_usable_table_names())

['scenario1']


<>:1: SyntaxWarning: invalid escape sequence '\A'
<>:1: SyntaxWarning: invalid escape sequence '\A'
C:\Users\Dongheekoh\AppData\Local\Temp\ipykernel_29008\2361953050.py:1: SyntaxWarning: invalid escape sequence '\A'
  db = SQLDatabase.from_uri("sqlite:///D:\AI_team\sqlLite\devDB.db")


In [138]:
db.run("select * from scenario1;")

"[('시간', '아군 전술', '적군 x좌표', '적군 y좌표', '적군 방향', '적군 이동속도', '적군 가시거리', '적군 탐지거리', '적군 전파상태', '적군 체력'), ('1', '1', '1020', '420', '270', '10', '20', '100', '5', '100'), ('2', '2', '1010', '418', '280', '10', '20', '100', '4', '100'), ('3', '2', '1003', '418', '275', '8', '20', '100', '4', '100'), ('4', '3', '1000', '417', '270', '6', '20', '100', '4', '100'), ('5', '3', '998', '416', '270', '4', '20', '100', '4', '95'), ('6', '3', '998', '416', '270', '0', '20', '30', '1', '95'), ('7', '4', '998', '416', '270', '0', '20', '30', '1', '75'), ('8', '4', '998', '416', '270', '0', '5', '30', '1', '75'), ('9', '4', '998', '416', '270', '0', '5', '30', '1', '75'), ('10', '5', '998', '416', '270', '0', '5', '30', '1', '75')]"

In [54]:
llm = ChatOpenAI(
            api_key="ai",
            model="openai/gpt-oss-20b",
            base_url="http://192.168.0.108:8000/v1",
        )

In [55]:
sql_query_gen_chain = create_sql_query_chain(llm, db)

In [162]:
clean_sql_prompt_template = """You are an expert in SQL Lite. You are asked to fix badly formedSQL Lite queries,
which might contain unneeded prefixes or suffixes. Given the following unclean SQL statement, transform it to a clean, 
excutable SQL statement for SQL lite. 
Only return an executable SQL statement which terminates with a semicolon. 
Do not return anything else. Do not include the language name or symbol like ```.

Unclean SQL: {unclean_sql}"""

remake_prompt_template = """
예시를 참고해서 질문에 대한 답변을 생성해줘

==예시==
질문: 시간이 3까지 흘렀을 때 현재 상황에 대해 묘사해줘
답변: 첫번째 행 전체와 첫번째 열 값이 1부터 3까지의 행 값을 출력해줘

질문: 시간이 5까지 흘렀을 때 현재 상황에 대해 묘사해줘
답변: 첫번째 행 전체와 첫번째 열 값이 1부터 5까지의 행 값을 출력해줘

질문: 시간이 3에서 8까지 흘렀을 때 현재 상황에 대해 묘사해줘
답변: 첫번째 행 전체와 첫번째 열 값이 3부터 8까지의 행 값을 출력해줘
========

질문: {question}

그리고 답변은 dict형식으로 "question": 답변 
이렇게 답변을 생성해줘
생성된 답변을 RunnablePassthrough로 다음 chain에 넘길거니까 오류 나지 않도록 형식에 맞춰서 답변해줘
"""

clean_sql_prompt = ChatPromptTemplate.from_template(clean_sql_prompt_template)

remake_ques_prompt = ChatPromptTemplate.from_template(remake_prompt_template)

clean_sql_chain = clean_sql_prompt | llm

remake_chain = remake_ques_prompt | llm | StrOutputParser()

full_sql_gen_chain = sql_query_gen_chain | clean_sql_chain | StrOutputParser()

sql_query_exec_chain = QuerySQLDataBaseTool(db=db)

sql_query_gen_and_exec_chain = full_sql_gen_chain | sql_query_exec_chain | StrOutputParser()

In [165]:
question = "첫번째 행 전체를 출력해줘"

res = remake_chain.invoke({"question": question})
print(res)

{"question":"첫번째 행 전체를 출력해줘"}


In [166]:
res3 = full_sql_gen_chain.invoke({"question": res})

print(res3)

SELECT field1, field2, field3, field4, field5, field6, field7, field8, field9, field10 FROM scenario1 ORDER BY field1 ASC LIMIT 1;


In [169]:
response = sql_query_gen_and_exec_chain.invoke({"question": "첫번째 행 전체를 출력해줘"})

response.extend()
result = sql_query_gen_and_exec_chain.invoke({"question": res})


In [170]:
print(response)

[('시간', '아군 전술', '적군 x좌표', '적군 y좌표', '적군 방향', '적군 이동속도', '적군 가시거리', '적군 탐지거리', '적군 전파상태', '적군 체력')]


In [148]:
response = [('시간', '아군 전술', '적군 x좌표', '적군 y좌표', '적군 방향', '적군 이동속도', '적군 가시거리', '적군 탐지거리', '적군 전파상태', '적군 체력'), ('1', '1', '1020', '420', '270', '10', '20', '100', '5', '100'), ('2', '2', '1010', '418', '280', '10', '20', '100', '4', '100'), ('3', '2', '1003', '418', '275', '8', '20', '100', '4', '100'), ('4', '3', '1000', '417', '270', '6', '20', '100', '4', '100'), ('5', '3', '998', '416', '270', '4', '20', '100', '4', '95'), ('6', '3', '998', '416', '270', '0', '20', '30', '1', '95'), ('7', '4', '998', '416', '270', '0', '20', '30', '1', '75'), ('8', '4', '998', '416', '270', '0', '5', '30', '1', '75'), ('9', '4', '998', '416', '270', '0', '5', '30', '1', '75'), ('10', '5', '998', '416', '270', '0', '5', '30', '1', '75')]


In [149]:
res = {"question":"현재 전장상황에 대해 묘사해줘"}

In [152]:
prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    """
                    너는 군사 도메인 전문가야
                    시나리오는 아군(북한군 KPA)과 적군(한국군 ROK)의 교전상황이고,
                    
                    ==아군 전술==
                    전술번호 | 전술이름 | 전술설명
                    1 | plan | AO() 상황 인식 및 정지 목표 식별
                    2 | prepare | 정찰/감시 및 제압 부대 조직, 장애물 준비
                    3 | execute-prevent-move | 적 이동 방해/지연, 보안 부대 격파
                    4 | execute-fix | 직접/간접 사격 및 EW(전자전)로 적 고착
                    5 | continue-mission | 임무 계속 및 ROK 재구성 방해
                    
                    다음 데이터는 열별로 다음 값을 의미해
                    열1: 시간
                    열2: 아군 전술
                    열3: 적군x좌표
                    열4: 적군y좌표
                    열5: 적군 방향
                    열6: 적군 이동속도
                    열7: 적군 가시거리
                    열8: 적군 탐지거리
                    열9: 적군 전파상태(0나쁨, 5최상)
                    열10: 적군 체력

                    테이블 정보: {response}

                    질문과 이 정보들을 바탕으로 시간흐름에 따른 현재 상황에 대해 자세히 묘사해줘
                    """
                ),
                ("human", "{question}"),
            ]
        )
chain = (
    prompt
    | llm
)

In [153]:
result = chain.invoke({"response": response, "question": res})
print(result.content)

**전장 상황 개요 – 시간 흐름에 따른 상세 묘사**

| 시간 | 아군 전술 | 적군 좌표 (X/Y) | 적군 방향 | 속도 | 가시거리 | 탐지거리 | 전파상태 | 체력 | 주목 포인트 |
|------|------------|-------------------|-----------|------|----------|--------|----------|------|------------|
| 1초  | **1 – plan**  | (1020, 420) | 270° (남쪽) | 10 m / s | 20 m | 100 m | 5 | 100 | AO() 상황 인식 완료. 전공을 명확히했으며, 전방(남쪽)을 정지 목표선으로 설정. |
| 2초  | **2 – prepare** | (1010, 418) | 280° | 10 m / s | 20 m | 100 m | 4 | 100 | 부대 재배치가 시작. 방어선 및 장애물(명중지상 포) 배치 진행. |
| 3초  | **2 – prepare** | (1003, 418) | 275° | 8 m / s | 20 m | 100 m | 4 | 100 | 이동 속도가 조금 감소. 남쪽으로 가파르게 전진. |
| 4초  | **3 – execute‑prevent‑move** | (1000, 417) | 270° | 6 m / s | 20 m | 100 m | 4 | 100 | 전방 부대가 적군 이동 방해(무장 장비)와 제압비(전력동원). 적군 전진 속도는 감소. |
| 5초  | **3 – execute‑prevent‑move** | (998, 416) | 270° | 4 m / s | 20 m | 100 m | 4 | 95 | 전파상태 4→1, 장비에 전파간섭이 발생. 적의 무장·정찰이 약화되며 체력이 5점 감소. |
| 6초  | **4 – execute‑fix** | (998, 416) | 270° | 0 m / s | 20 m | 30 m | 1 | 95 | 적군이 완전히 정지. EW(전자전)로 방향 고정 및 반격 방지.